# Figures 2-3 - Reconstruction

Generates mobility-distribution and reconstruction/simulation diagnostics.

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm

PALETTE = {
    "src_dest": "#E7D1D1",
    "rnd_rnd": "#DCC7DF",
    "rnd_dest": "#C4DADD",
    "src_rnd": "#C8DEC8",

    "roads_main": "#8D8D8D",
    "roads_minor": "#CFCFCF",

    "text": "#2F2F2F",
    "background": "#FAFAFA",
    "grid": "#DDDDDD",
}

paper_cmap = LinearSegmentedColormap.from_list(
    "paper_activity",
    [
        "#EAF6F7",  # very low
        "#AFCFD4",  # low
        "#D5C0DC",  # medium
        "#D9B8B8",  # high
        "#9E5F63",  # very high
    ],
    N=256,
)

# ---------- GLOBAL MATPLOTLIB STYLE ----------
plt.rcParams.update({

    # Figure
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["background"],
    "savefig.facecolor": PALETTE["background"],

    # Text
    "text.color": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],

    # Font sizes
    "font.size": 18,
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 26,

    # Axes
    "axes.edgecolor": "#BDBDBD",
    "axes.linewidth": 1.2,
    "axes.grid": False,

    # Grid
    "grid.color": PALETTE["grid"],
    "grid.alpha": 0.4,
    "grid.linewidth": 0.8,

    # Legend
    "legend.frameon": True,
    "legend.facecolor": "white",
    "legend.edgecolor": "#DDDDDD",

    # Lines
    "lines.linewidth": 2.2,

    # Ticks
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,

    # DPI
    "figure.dpi": 300,
    "savefig.dpi": 300,

    # PDF/SVG publication quality
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

In [2]:
# Common imports and output utility
# Source: requested notebook standard; run from repository root.
try:
    import os
    from pathlib import Path
    import warnings
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.colors import LogNorm
    import seaborn as sns
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import contextily
    from PIL import Image

    start_dir = Path.cwd().resolve()
    ROOT = next((candidate for candidate in [start_dir, *start_dir.parents]
                 if (candidate / "article/tex/epjds_main_draft.tex").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Repository root not found from current working directory")
    os.chdir(ROOT)
    images_dir = Path("article/tex/images")
    images_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)
    plt.rcParams.update({"savefig.dpi": 300, "pdf.fonttype": 42, "ps.fonttype": 42})

    def save_fig_local(fig, stem):
        # Write the manuscript target as both raster and vector output.
        stem = Path(stem).stem
        for ext in ("png", "pdf"):
            fig.savefig(images_dir / f"{stem}.{ext}", bbox_inches="tight", dpi=300)
        plt.close(fig)
        print(f"OK: saved {stem}.png and {stem}.pdf in {images_dir}")

    print(f"OK: setup complete; repo root is {ROOT}; image output is {images_dir.resolve()}")
except Exception as exc:
    print(f"ERROR setup: {type(exc).__name__}: {exc}")
    raise


OK: setup complete; repo root is /mnt/c/Users/belli/OneDrive/Desktop/code/covid_hep; image output is /mnt/c/Users/belli/OneDrive/Desktop/code/covid_hep/article/tex/images


In [3]:
# Shared loaders for OD temporal profiles (Figures 2a-2b) and simulation outputs.
# OD source: article/ipynb/od.ipynb cells 7, 13 and 16.
try:
    od_source = ROOT.parent / "scaling_fundamental_diagram" / "dataset" / "ER" / "ER_data_raw.csv"

    def load_od_time_profiles():
        if not od_source.exists():
            raise FileNotFoundError(f"OD input not found: {od_source}")
        grouped_chunks = []
        columns = ["date", "time", "ACE_orig", "ACE_dest", "N"]
        for chunk in pd.read_csv(od_source, usecols=columns, chunksize=500_000):
            stamp = pd.to_datetime(chunk["date"] + " " + chunk["time"], errors="coerce")
            origin = chunk["ACE_orig"].astype(str).str.replace("C", "", regex=False)
            destination = chunk["ACE_dest"].astype(str).str.replace("C", "", regex=False)
            selected = (
                origin.str.startswith("0080")
                & destination.str.startswith("0080")
                & (origin != destination)
                & (stamp > pd.Timestamp("2020-01-01 00:00:00"))
                & (stamp <= pd.Timestamp("2020-01-31 23:00:00"))
            )
            flows = pd.DataFrame({
                "datetime": stamp[selected],
                "municipality_code": origin[selected],
                "N": chunk.loc[selected, "N"],
            })
            grouped_chunks.append(flows.groupby(["datetime", "municipality_code"])["N"].sum())
        flows = pd.concat(grouped_chunks).groupby(level=[0, 1]).sum().reset_index()
        flows["time"] = flows["datetime"].dt.time
        flows["day_type"] = np.where(flows["datetime"].dt.dayofweek < 5, "weekday", "weekend")
        profiles = flows.groupby(["time", "day_type"])["N"].mean().mul(3).unstack()
        profiles.index = pd.to_datetime([f"2023-06-30 {time}" for time in profiles.index])
        return profiles["weekday"], profiles["weekend"]

    sim_dir = Path("article/data/uniq")

    def simulation_files(year):
        files = sorted(sim_dir.glob(f"*_uniq_{str(year)[2:]}*.csv.corrected"))
        if not files:
            files = sorted(sim_dir.glob(f"*_uniq_{str(year)[2:]}*.csv"))
            print(f"INFO: {year}: corrected files absent; using raw uniq output (TODO confirm).")
        return files

    def load_sim_profile(year):
        rows = []
        for path in simulation_files(year):
            frame = pd.read_csv(path, sep=";")
            hourly = [c for c in frame.columns if c.startswith(str(year))]
            if not hourly:
                continue
            grouped = frame.groupby("poly_cid")[hourly].sum()
            profile = grouped.sum(axis=0)
            profile.index = pd.to_datetime(profile.index).hour
            rows.append(profile.groupby(level=0).sum().reindex(range(24), fill_value=0))
        if not rows:
            raise FileNotFoundError(f"No simulation files for {year} in {sim_dir}")
        return pd.DataFrame(rows).mean(axis=0)

    print("OK: OD temporal and simulation profile loaders ready")
except Exception as exc:
    print(f"ERROR simulation loader: {type(exc).__name__}: {exc}")


OK: OD temporal and simulation profile loaders ready


In [4]:
# 2a_weekday_reconstruction.png - mean weekday OD activity profile.
# Recreates the upper panel of article/ipynb/od.ipynb cell 16.
try:
    od_weekday, od_weekend = load_od_time_profiles()
    fig, ax = plt.subplots(figsize=(7.2, 4.2), constrained_layout=True)
    ax.plot(od_weekday.index, od_weekday.values, "-o", color= PALETTE["roads_main"], label="Week Days", markersize=4, lw=1.8)
    ax.plot(od_weekend.index, od_weekend.values, "-o", color=PALETTE["roads_minor"], label="Weekend Days", markersize=4, lw=1.8)
    
    ax.set_ylabel("Vehicles / hour")
    ax.set_xlabel("Time")
    ticks = od_weekday.index[::3]
    ax.set_xticks(ticks)
    ax.set_xticklabels([time.strftime("%H:%M") for time in ticks])
    # ax.set_xlim(od_weekday.index.min(), od_weekday.index.max())
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.grid(False)
    ax.legend()
    save_fig_local(fig, "2a_week_reconstruction")
except Exception as exc:
    print(f"ERROR 2a_weekday_reconstruction: {type(exc).__name__}: {exc}")
    raise


OK: saved 2a_week_reconstruction.png and 2a_week_reconstruction.pdf in article/tex/images


In [5]:
# 2b/2d mobility pies - weekday and weekend mobility-type distributions.
# Source: work_viaggi/mob_type; colors are defined in the palette cell above.
try:
    mobility_files = sorted(Path("work_viaggi/mob_type").glob("*.csv"))
    if not mobility_files:
        raise FileNotFoundError("work_viaggi/mob_type/*.csv")
    mobility = pd.concat([pd.read_csv(path, sep=";") for path in mobility_files], ignore_index=True)
    mobility["Mobility_Type"] = mobility["Mobility_Type"].astype(str).str.strip().str.replace("_", "-", regex=False)
    mobility["day_type"] = np.where(pd.to_datetime(mobility["day"]).dt.dayofweek < 5, "Weekday", "Weekend")

    mobility_order = ["src-dest", "src-rnd", "rnd-dest", "rnd-rnd"]
    colors = [
        PALETTE["src_dest"], PALETTE["src_rnd"],
        PALETTE["rnd_dest"], PALETTE["rnd_rnd"],
    ]

    def plot_mobility_pie(day_type, stem):
        values = (
            mobility.loc[mobility["day_type"] == day_type]
            .groupby("Mobility_Type")["Percentage"].mean()
            .reindex(mobility_order)
        )
        fig, ax = plt.subplots(figsize=(8.4, 7.0), dpi=300, constrained_layout=True)
        wedges, labels, percentages = ax.pie(
            values, labels=mobility_order, colors=colors, autopct="%1.1f%%",
            startangle=120, radius=1.0, labeldistance=1.08, pctdistance=0.62,
            wedgeprops={"linewidth": 1.5, "edgecolor": "white"},
            textprops={"fontsize": 22, "fontweight": "bold", "color": PALETTE["text"]},
        )
        for label in labels + percentages:
            label.set_fontsize(22)
            label.set_fontweight("bold")
        ax.set_title(f"{day_type} mobility distribution", fontsize=28, fontweight="bold", pad=18)
        save_fig_local(fig, stem)

    plot_mobility_pie("Weekday", "2b_mobility_weekday")
    plot_mobility_pie("Weekend", "2d_mobility_weekend")
except Exception as exc:
    print(f"ERROR mobility_pies: {type(exc).__name__}: {exc}")
    raise


OK: saved 2b_mobility_weekday.png and 2b_mobility_weekday.pdf in article/tex/images
OK: saved 2d_mobility_weekend.png and 2d_mobility_weekend.pdf in article/tex/images


In [6]:
# Mobile activity ranking: first and last ten municipalities within the top 50.
# Source: MCF/data/derived/city_activity_top50.csv; reference figure: mobile_activity_top15.png.
try:
    from matplotlib.patches import Patch

    city_activity = pd.read_csv("MCF/data/derived/city_activity_top50.csv").sort_values("mean_points", ascending=False)
    shown_cities = city_activity.head(15).assign(group="Top 15")
    bar_colors = shown_cities["group"].map({"Top 15": PALETTE["src_dest"]})

    fig, ax = plt.subplots(figsize=(9.5, 18.2), constrained_layout=True)
    ax.barh(shown_cities["name"], shown_cities["mean_points"], color=bar_colors, edgecolor="white", linewidth=1.2)
    ax.invert_yaxis()
    # ax.set_title("Mobile activity: highest and lowest ranked cities", fontsize=25, fontweight="bold", pad=16)
    ax.set_xlabel("Mean active points", fontsize=24, fontweight="bold")
    ax.tick_params(axis="both", labelsize=26)
    ax.tick_params(axis="x", labelrotation=45)
    ax.xaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
    ax.grid(False)
    ax.set_axisbelow(True)
    # ax.legend(handles=[Patch(facecolor=PALETTE["src_dest"], label="Top 15")],
    #           loc="lower right", frameon=True)
    save_fig_local(fig, "3_mobile_activity_top_bottom10")
except Exception as exc:
    print(f"ERROR mobile_activity_ranking: {type(exc).__name__}: {exc}")
    raise


OK: saved 3_mobile_activity_top_bottom10.png and 3_mobile_activity_top_bottom10.pdf in article/tex/images


In [27]:
# Hourly comparison between corrected simulation and observed coil counts.
# Coil 282 maps to poly_lid 2107, corresponding to poly_cid 214686 in this simulation.
try:
    sim_path = Path("article/data/uniq/sim_230330113558146127_uniq_200210_000000.csv.corrected")
    coil_path = Path("data/old_coils_data/_coils_table_200201-000000_200331-234500.csv")
    selected_poly = 214686
    selected_coil = 282
    selected_day = pd.Timestamp("2020-02-10")
    simulation = pd.read_csv(sim_path, sep=";")
    hourly_columns = [column for column in simulation.columns if column.startswith("2020-")]
    poly_flow = simulation.loc[simulation["poly_cid"] == selected_poly, hourly_columns].sum(axis=0)
    if poly_flow.empty or poly_flow.sum() == 0:
        raise ValueError(f"No positive simulated flow found for poly_cid {selected_poly}")
    poly_flow.index = pd.to_datetime(poly_flow.index)
    coil = pd.read_csv(coil_path, sep=";", usecols=["datetime", "282_0", "282_1"], parse_dates=["datetime"])
    coil["datetime"] = coil["datetime"].dt.tz_convert(None)
    daily_coil = coil.loc[(coil["datetime"] >= selected_day) & (coil["datetime"] < selected_day + pd.Timedelta(days=1))].copy()
    if len(daily_coil) != 96:
        raise ValueError(f"Expected 96 quarter-hour observations for coil {selected_coil}; found {len(daily_coil)}")
    daily_coil["flow"] = daily_coil[["282_0", "282_1"]].sum(axis=1, min_count=1)
    coil_flow = daily_coil.set_index("datetime")["flow"]#.resample("h").sum(min_count=1)

    fig, ax = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
    ax.plot(poly_flow.index, poly_flow.values, color=PALETTE["roads_main"], marker="o", markersize=6, linewidth=2.7,
            label=f"Simulation")
    ax.plot(coil_flow.index, coil_flow.values * 4, color=PALETTE["roads_minor"], marker="s", 
            markersize=5.5, linewidth=2.4, linestyle="--", markevery=4, label=f"Coil {selected_coil}")
    # ax.set_title("Hourly flow comparison: coil 282 and mapped polyline", fontsize=24, fontweight="bold", pad=14)
    ax.set_xlabel("Time", fontsize=19)
    ax.set_ylabel(r'$\Phi$', fontsize=19, fontweight="bold", rotation=0)
    ax.tick_params(axis="both", labelsize=16)
    ax.tick_params(axis="x", labelrotation=45, labelsize=16)
    ax.xaxis.set_major_locator(matplotlib.dates.HourLocator(interval=3))
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%H:%M"))
    ax.spines[['right', 'top']].set_visible(False)
    ax.legend(loc="upper right", frameon=True, fontsize=18)

    ax.grid(False)
    # ax.set_axisbelow(True)
    save_fig_local(fig, "4_poly_flow")
    print(f"OK: compared hourly simulated polyline {selected_poly} with observed coil {selected_coil} on {selected_day.date()}")
except Exception as exc:
    print(f"ERROR selected_poly_flow: {type(exc).__name__}: {exc}")
    raise


OK: saved 4_poly_flow.png and 4_poly_flow.pdf in article/tex/images
OK: compared hourly simulated polyline 214686 with observed coil 282 on 2020-02-10


In [31]:
# Static PNG/PDF version of the original folium fluxes_map style
try:
    import numpy as np
    import geopandas as gpd
    import matplotlib.pyplot as plt
    from matplotlib import cm
    from matplotlib.colors import rgb2hex
    from matplotlib.lines import Line2D
    import contextily as ctx

    # --- Load network and simulated flow ---
    network = gpd.read_file("article/data/uniq/carto.geojson")

    daily_flow = (
        simulation
        .groupby("poly_cid")[hourly_columns]
        .sum()
        .sum(axis=1)
        .rename("uniq_cnt")
    )

    map_data = network.merge(daily_flow, on="poly_cid", how="left")
    map_data["uniq_cnt"] = map_data["uniq_cnt"].fillna(0)

    # --- Same quantile logic as original script ---
    quantiles_val = [0.0, 0.20, 0.40, 0.60, 0.80, 0.90, 1.0]
    quantiles = map_data["uniq_cnt"].quantile(quantiles_val).to_list()
    bins = sorted(set(int(q) for q in quantiles))

    # Avoid pd.cut errors if duplicated quantiles collapse bins
    if len(bins) < 2:
        bins = [0, max(1, int(map_data["uniq_cnt"].max()))]

    cmap = cm.get_cmap("viridis", len(bins) - 1)
    rgb_scale = cmap(np.linspace(0, 1, len(bins)))
    color_scale = [rgb2hex(r) for r in rgb_scale][:-1]

    default_color = color_scale[1] if len(color_scale) > 1 else color_scale[0]

    map_data["color"] = pd.cut(
        map_data["uniq_cnt"],
        bins=bins,
        labels=color_scale,
        include_lowest=True,
        duplicates="drop"
    )

    map_data["color"] = map_data["color"].astype("object").fillna(default_color)

    # Original script draws zero-flow links in red
    map_data.loc[map_data["uniq_cnt"] == 0, "color"] = "#FF0000"

    # --- Project for basemap ---
    map_web = map_data.to_crs(epsg=3857)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(13.0, 6.5), constrained_layout=True)

    # Plot by color, like folium FeatureGroups
    for col, grp in map_web.groupby("color"):
        grp.plot(
            ax=ax,
            color=col,
            linewidth=1.05 if col != "#FF0000" else 0.45,
            alpha=0.95 if col != "#FF0000" else 0.55,
            zorder=3 if col != "#FF0000" else 2,
        )

    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        attribution=False,
        zoom=8
    )

    ax.set_axis_off()

    # --- Legend similar to quantile layers ---
    handles = []

    for i in range(len(bins) - 1):
        handles.append(
            Line2D(
                [0], [0],
                color=color_scale[i],
                lw=3,
                label=f"{quantiles_val[i]:.2f} < q ≤ {quantiles_val[i+1]:.2f}"
            )
        )

    handles.append(
        Line2D([0], [0], color="#FF0000", lw=3, label="zero flow")
    )

    ax.legend(
        handles=handles,
        title="Daily flow quantiles",
        loc="lower left",
        fontsize=11,
        title_fontsize=12,
        frameon=True
    )

    save_fig_local(fig, "5_simulated_daily_flux_map")

    fig.savefig("/home/fede/code/covid_hep/article/tex/images/5_simulated_daily_flux_map.png", dpi=300, bbox_inches="tight")
    fig.savefig("/home/fede/code/covid_hep/article/tex/images/5_simulated_daily_flux_map.pdf", bbox_inches="tight")

except Exception as exc:
    print(f"ERROR simulation_flow_map: {type(exc).__name__}: {exc}")
    raise

/tmp/ipykernel_26889/92981337.py:34: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("viridis", len(bins) - 1)


OK: saved 5_simulated_daily_flux_map.png and 5_simulated_daily_flux_map.pdf in article/tex/images
